# SAR-Assisted Understanding — analysis walkthrough

Reads the artefacts written by `scripts/03_run_experiments.py` and `04_analyse.py`.
Run the pipeline first (`bash scripts/run_experiment.sh`), then execute this top to bottom.

**Question.** Does Sentinel-1 recover land-cover accuracy lost when Sentinel-2 is
partially obscured? The comparison that matters is **B vs C** at each masking level.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import load_config
from src.visualization import use_style
use_style()
M = ROOT / 'results' / 'metrics'
cfg = load_config(ROOT / 'configs' / 'config.yaml')
print('levels:', cfg.masking.levels, '| seeds:', cfg.train.seeds, '| threshold:', cfg.eval.threshold)


## 1. The data selection

BigEarthNet v2.0 is multi-label; we keep the 11 classes with >= 1000 positive patches.


In [ ]:
summary = json.load(open(M / 'subset_summary.json'))
manifest = pd.read_csv(M / 'subset_manifest.csv')
print(f"{summary['n_patches']} patches, {len(summary['classes'])} classes")
print('splits:', summary['split_sizes'])
print('audit problems:', summary['n_audit_problems'], 'of', summary['n_audited'], 'pairs checked')
pd.Series(summary['class_positives']).sort_values(ascending=False).to_frame('positives')


### Splits are spatially blocked, not random

Test is a contiguous interior region ringed by validation, ringed by train — so
neighbouring (highly correlated) patches almost never straddle train and test.


In [ ]:
sym = {'train': 0, 'validation': 1, 'test': 2}
fig, axes = plt.subplots(1, manifest.tile.nunique(), figsize=(11, 4.2))
for ax, (tile, g) in zip(np.atleast_1d(axes), manifest.groupby('tile')):
    grid = np.full((g.row.max()+1, g.col.max()+1), np.nan)
    grid[g.row, g.col] = g.split.map(sym)
    ax.imshow(grid, cmap='viridis', interpolation='nearest')
    ax.set_title(f'{tile}  (n={len(g)})'); ax.grid(False)

    idx = {(r, c): s for r, c, s in zip(g.row, g.col, g.split)}
    cross = tot = 0
    for (r, c), s in idx.items():
        for dr, dc in ((0, 1), (1, 0)):
            n = idx.get((r+dr, c+dc))
            if n is not None:
                tot += 1; cross += n != s
    print(f'{tile}: {cross}/{tot} adjacent pairs cross a split boundary ({cross/tot:.1%})')
plt.suptitle('Split assignment on the patch grid (dark=train, mid=val, light=test)')
plt.tight_layout(); plt.show()


## 2. The degradation is spatially coherent and exact

Masks are a deterministic function of `(patch_id, level, seed)`, so **arms B and C see
byte-identical degraded imagery** — the property the whole comparison rests on.


In [ ]:
from src.masking import generate_mask
levels = [float(v) for v in cfg.masking.levels]
fig, axes = plt.subplots(1, len(levels), figsize=(15, 2.7))
for ax, lv in zip(axes, levels):
    m = generate_mask('S2B_MSIL2A_20170808T094029_N9999_R036_T35ULA_20_20', lv,
                      sigma=cfg.masking.smoothing_sigma, mask_seed=cfg.masking.mask_seed,
                      mode=cfg.masking.mode)
    ax.imshow(m, cmap='gray_r'); ax.set_title(f'{lv:.0%}  (actual {m.mean():.1%})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.suptitle('Simulated cloud masks — coverage is exact by construction'); plt.tight_layout(); plt.show()


## 3. Headline result

Two regimes. **R1** trains on clean optical and evaluates degraded (the assignment's
minimum baseline). **R2** trains *both* arms with the same masking augmentation, so the
only difference between them is the SAR branch.


In [ ]:
gain = pd.read_csv(M / 'sar_gain_table.csv')
cols = ['masking_pct', 'B_degraded_optical', 'C_degraded_plus_sar', 'sar_gain',
        'sar_gain_boot_lo', 'sar_gain_boot_hi', 'control_fusion_shuffled_sar', 'D_sar_only']
for regime in ['clean', 'degraded']:
    print(f'--- {regime} ---')
    display(gain[gain.regime == regime][cols].set_index('masking_pct').round(4))


**Read this carefully.** In R1 the SAR gain climbs to +0.27 at 80% masking. In R2 it is
+0.037 at the same level. Most of the apparent benefit of SAR is actually the benefit of
*training on degraded data* — a confound that a B-vs-C comparison alone would hide.


In [ ]:
res = pd.read_csv(M / 'results.csv')
res.pivot_table(index=['regime', 'test_level'], columns='arm',
                values='macro_f1', aggfunc=['mean', 'std']).round(4)


### The capacity control

`fusion_shuf` is arm C with SAR features permuted across samples: identical parameter
count, no genuine pairing. It scores *below* optical-only everywhere, so arm C's
behaviour is not explained by having more parameters.


In [ ]:
d = res[res.regime == 'degraded']
for arm in ['optical', 'fusion', 'fusion_shuf', 'sar']:
    g = d[d.arm == arm].groupby('test_level').macro_f1.mean()
    plt.plot(g.index * 100, g.values, marker='o', label=arm)
plt.xlabel('% masked'); plt.ylabel('macro F1'); plt.legend(); plt.title('R2: all arms'); plt.show()


### Threshold-free view

Macro F1 at 100% collapses to ~0 because a constant input yields the class prior, which
sits below the 0.5 threshold. mAP shows what is actually retained.


In [ ]:
res[res.regime == 'degraded'].pivot_table(index='test_level', columns='arm',
                                          values='macro_map', aggfunc='mean').round(4)


## 4. SAR does not help all classes equally


In [ ]:
pcg = pd.read_csv(M / 'per_class_sar_gain.csv')
piv = pcg[pcg.regime == 'degraded'].pivot_table(index='class', columns='masking_pct', values='sar_gain')
piv.sort_values(80, ascending=False).round(3)


Inland waters gains at *every* level (calm water is a specular reflector — near-black and
unambiguous in SAR). Broad-leaved forest and Pastures are hurt at every operational level:
backscatter cannot separate canopy types that reflectance can.


## 5. Failure analysis

Cases are chosen by a rule fixed in advance, with both failure categories included by
construction. At 80% masking SAR repairs ~1.7% of patches and breaks ~3.0% — yet macro F1
still improves, because the repairs concentrate in rare classes that macro-averaging weights.


In [ ]:
from src.visualization import sample_f1
z = np.load(M / 'test_probs.npz', allow_pickle=True)
thr = float(cfg.eval.threshold)
t = z['targets']
pb, pc = z['degraded/optical/0.8'], z['degraded/fusion/0.8']
fb, fc = sample_f1(t, (pb >= thr).astype(int)), sample_f1(t, (pc >= thr).astype(int))
delta = fc - fb
print(f'SAR repairs : {(delta > 0.3).mean():.1%} of test patches')
print(f'SAR breaks  : {(delta < -0.3).mean():.1%}')
print(f'no change   : {(np.abs(delta) <= 0.3).mean():.1%}')
plt.hist(delta, bins=60, color='#eb6834'); plt.axvline(0, color='k', lw=1)
plt.xlabel('per-sample F1(C) - F1(B) at 80% masking'); plt.ylabel('patches'); plt.show()
pd.read_csv(M / 'qualitative_examples.csv')


## 6. Conclusion

1. Optical degradation is devastating for a model that never saw it (0.70 -> 0.28 at 80%)
   and mild for one that did (0.69 -> 0.60).
2. Against the fair baseline, SAR helps **only past ~60% cloud** (+0.037 at 80%), and is
   slightly *harmful* below that.
3. The benefit is concentrated in classes with distinctive backscatter — above all
   Inland waters (+0.17 to +0.23 at every level).
4. The hypothesis is therefore **partially supported**: SAR is complementary, but far less
   than the naive comparison suggests, and simple degradation-aware training buys about
   nine times more at 80% masking than SAR does.

See the README for limitations — chiefly that this subset spans only two Sentinel-2
acquisitions, which bounds how far any of these numbers generalise.
